# RG3 미성형 과다경보 D 물리확인/지원범위 수정


In [ ]:
from pathlib import Path
import os,json,nbformat,joblib
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();WORK=ROOT/'rg3/internal';tables=WORK/'reports/tables'
package=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4);exec(package.cells[1].source)
prev=Path.cwd()
try:
 os.chdir(WORK)
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb');base={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],base)
finally:os.chdir(prev)
lab=base['lab'];y=base['yn'];g=base['gn'];use=base['usable'];normal=use&(y==0)
dense=base['nc'][0]['dense'];hopper=-dense.Hopper_Temperature__dev10.to_numpy()
physical=base['nb']['views']['physical12'];joint=base['nd']['jointbase']
mechanical_indices=[0,1,2,3,4,5,6,7,8,10,11]
rawtemp=[f'Barrel_Temperature_{i}' for i in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
context=lab[[v+'_raw' for v in rawtemp]].to_numpy(float)
oldpred=pd.read_csv(tables/'rg3_204_shortshot_reduction_predictions.csv',parse_dates=['TimeStamp'])
oldpred=oldpred[oldpred.candidate.eq('pruned15')].set_index('TimeStamp').loc[lab.TimeStamp]
unknown=pd.read_csv(tables/'rg3_207_dense_unknown_history.csv',parse_dates=['TimeStamp'])
diag=pd.read_csv(tables/'rg3_207_shortshot_alarm_diagnostics.csv',parse_dates=['TimeStamp'])
u=unknown.set_index('TimeStamp').loc[diag.TimeStamp].reset_index()
ratio=lambda a,b:a/b.where(b.abs()>1e-6)
phyU=np.column_stack([u.Max_Injection_Pressure-u.Max_Switch_Over_Pressure,ratio(u.Max_Switch_Over_Pressure,u.Max_Injection_Pressure),ratio(u.Filling_Time,u.Injection_Time),ratio(u.Filling_Time,u.Max_Injection_Speed),u.Plasticizing_Position-u.Cushion_Position,u.Plasticizing_Position__delta1-u.Cushion_Position__delta1,ratio(u.Plasticizing_Time,u.Cycle_Time),u.Plasticizing_Time__sparse_dev,u.Average_Back_Pressure__sparse_dev,u.Barrel_Temperature_5-(u.Mold_Temperature_3+u.Mold_Temperature_4)/2,u.Max_Injection_Pressure__sparse_dev,u.Cushion_Position__sparse_dev])
jointU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Hopper_Temperature__past10_std,u.Max_Injection_Speed__past10_std,u.Cushion_Position__past10_std,u.Plasticizing_Time__past10_std,u.Barrel_Temperature_5__past10_std,u.Average_Back_Pressure__past10_std])
ctxU=u[rawtemp].to_numpy(float)
def mech_novelty(ref,qx):
 x=physical[ref][:,mechanical_indices];x=x[y[ref]==0]
 imp=SimpleImputer(strategy='median').fit(x);x=imp.transform(x)
 med=np.median(x,axis=0);scale=np.maximum(np.quantile(x,.75,axis=0)-np.quantile(x,.25,axis=0),np.maximum(x.std(axis=0),.001))
 score=lambda a:np.max(np.abs((imp.transform(a)-med)/scale),axis=1)
 train=score(x)
 return score(qx[:,mechanical_indices]),{quant:float(np.quantile(train,quant,method='higher')) for quant in [.90,.95,.99]}
def temp_support(ref,qcontext):
 x=context[ref][y[ref]==0]
 imp=SimpleImputer(strategy='median').fit(x);x=imp.transform(x)
 med=np.median(x,axis=0);scale=np.maximum(np.quantile(x,.75,axis=0)-np.quantile(x,.25,axis=0),.1)
 a=(x-med)/scale;b=(imp.transform(qcontext)-med)/scale
 nn=NearestNeighbors(n_neighbors=2).fit(a)
 calibration=nn.kneighbors(a)[0][:,1]
 cut=float(np.quantile(calibration,.99,method='higher'))
 dist=nn.kneighbors(b,n_neighbors=1)[0][:,0]
 return dist<=cut,dist,cut
score_cache={};fit_audit=[]
def d_scores(ref,query_joint,query_phy):
 # Fit scalers/normalization only on ref normals. Query quality labels are not used.
 jm=joint[ref];imp=SimpleImputer(strategy='median').fit(jm[y[ref]==0]);sc=RobustScaler().fit(imp.transform(jm[y[ref]==0]))
 a=np.clip(sc.transform(imp.transform(jm)),-12,12);b=np.clip(sc.transform(imp.transform(query_joint)),-12,12)
 def rel(x):
  v=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))
  return np.column_stack([v.max(axis=1),np.sort(v,axis=1)[:,-2],v.mean(axis=1),(v>1).sum(axis=1)])
 ra,rb=rel(a),rel(b)
 ip=SimpleImputer(strategy='median').fit(physical[ref][:,mechanical_indices][y[ref]==0]);sp=RobustScaler().fit(ip.transform(physical[ref][:,mechanical_indices][y[ref]==0]))
 pa=np.clip(sp.transform(ip.transform(physical[ref][:,mechanical_indices])),-12,12)
 pb=np.clip(sp.transform(ip.transform(query_phy[:,mechanical_indices])),-12,12)
 out={}
 for name,ta,tb in [('old_joint',ra,rb),('joint_mechanical',np.column_stack([ra,pa]),np.column_stack([rb,pb]))]:
  model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(ta,y[ref])
  out[name]=model.predict_proba(tb)[:,1]
 return out
records=[];audits=[]
policies=['baseline15','D_temp_supported','D_mech90','D_mech95','D_mech99','D_joint_mechanical']
for block in pd.unique(g):
 q=np.flatnonzero(g==block);ref=np.flatnonzero((g!=block)&use)
 abc=oldpred.iloc[q][['A','B','C']].any(axis=1).to_numpy();oldD=oldpred.iloc[q].D.to_numpy(bool)
 supported,dist,dc=temp_support(ref,context[q]);nov,cuts=mech_novelty(ref,physical[q])
 outer=d_scores(ref,joint[q],physical[q]);inner=np.full(len(ref),np.nan)
 for ib in pd.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=d_scores(ref[~mask],joint[ref[mask]],physical[ref[mask]])['joint_mechanical']
 dcut=float(np.quantile(inner[y[ref]==0],.995,method='higher'))
 h=hopper[ref];hcut=float(np.quantile(h[(y[ref]==0)&np.isfinite(h)],.95,method='higher'))
 ds={'baseline15':oldD,'D_temp_supported':oldD&supported,'D_mech90':oldD&(nov>cuts[.90]),'D_mech95':oldD&(nov>cuts[.95]),'D_mech99':oldD&(nov>cuts[.99]),'D_joint_mechanical':(outer['joint_mechanical']>=dcut)&(hopper[q]>=hcut)}
 for policy,dd in ds.items():
  for j,idx in enumerate(q):records.append(dict(policy=policy,TimeStamp=lab.TimeStamp.iloc[idx],type=lab.type.iloc[idx],actual=int(y[idx]),prediction=int(abc[j]|dd[j]),D=int(dd[j]),review_required=int(policy=='D_temp_supported' and oldD[j] and not supported[j] and not abc[j])))
 audits.append(dict(block=block,train=len(ref),query=len(q),overlap=len(set(lab.TimeStamp.iloc[ref])&set(lab.TimeStamp.iloc[q]))))
 print('D correction completed',block,flush=True)
pp=pd.DataFrame(records);metrics=[]
for policy,a in pp[pp.type.isin(['정상','미성형'])].groupby('policy'):
 tn,fp,fn,tp=confusion_matrix(a.actual,a.prediction,labels=[0,1]).ravel()
 metrics.append(dict(policy=policy,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn)),review_required=int(a.review_required.sum())))
mm=pd.DataFrame(metrics)
ref=np.flatnonzero(use);abc=diag[['A','B','C']].any(axis=1).to_numpy();oldD=diag.D.to_numpy(bool)
support,dist,dc=temp_support(ref,ctxU);nov,cuts=mech_novelty(ref,phyU)
outer=d_scores(ref,jointU,phyU);inner=np.full(len(ref),np.nan)
for ib in pd.unique(g[ref]):
 mask=g[ref]==ib;inner[mask]=d_scores(ref[~mask],joint[ref[mask]],physical[ref[mask]])['joint_mechanical']
dcut=float(np.quantile(inner[y[ref]==0],.995,method='higher'));h=hopper[ref];hcut=float(np.quantile(h[(y[ref]==0)&np.isfinite(h)],.95,method='higher'))
ds={'baseline15':oldD,'D_temp_supported':oldD&support,'D_mech90':oldD&(nov>cuts[.90]),'D_mech95':oldD&(nov>cuts[.95]),'D_mech99':oldD&(nov>cuts[.99]),'D_joint_mechanical':(outer['joint_mechanical']>=dcut)&(-u.Hopper_Temperature__dev10.to_numpy()>=hcut)}
um=[]
for policy,dd in ds.items():
 pr=abc|dd
 review=(oldD&~support&~abc) if policy=='D_temp_supported' else np.zeros(len(u),bool)
 um.append(dict(policy=policy,N=len(u),candidates=int(pr.sum()),alarm_percent=float(100*pr.mean()),review_required=int(review.sum())))
 pd.DataFrame({'TimeStamp':u.TimeStamp,'prediction':pr.astype(int),'review_required':review.astype(int)}).to_csv(tables/('rg3_208_unlabeled_'+policy+'.csv'),index=False)
pp.to_csv(tables/'rg3_208_D_correction_predictions.csv',index=False)
mm.to_csv(tables/'rg3_208_D_correction_metrics.csv',index=False)
pd.DataFrame(um).to_csv(tables/'rg3_208_D_correction_unlabeled_metrics.csv',index=False)
pd.DataFrame(audits).to_csv(tables/'rg3_208_D_correction_audit.csv',index=False)
assert all(a['overlap']==0 for a in audits)
print(mm.to_string(index=False));print(pd.DataFrame(um).to_string(index=False))
text='# RG3 미성형 D 과다경보 수정 실험\n\n라벨구간제외검증과동일비라벨6590샷. 기존ABC는204 OOF/207 fullfit출력을고정하고D만변경했다.날짜를제외하거나비라벨에정상라벨을붙이지않았다.\n\n'+mm.to_string(index=False)+'\n\n'+pd.DataFrame(um).to_string(index=False)
text+='\n\nD_temp_supported의review_required는훈련정상온도상태와다른곳에서기존D가경보한샷으로,정상처리한것이아니라보류/추가확인대상이다.라벨574샷은보류샷도자동경보음성으로포함하여FN을숨기지않았다.나머지물리확인게이트는새판정규칙의경보이지불량정답확인이아니다.새독립검증아닌이미본평가자료의개발비교이며기존배포파일/Git미교체.\n'
(WORK/'reports/rg3_208_D_correction_review.md').write_text(text,encoding='utf-8')

